In [3]:
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
import umap

DATA_DIR = Path("Outputs/PacketEmbeddingsMLM")
SEED = 42

# ---- load ----
X_parts, labels = [], []
for f in sorted(DATA_DIR.glob("*.npy")):
    arr = np.load(f)
    X_parts.append(arr)
    labels += [f.stem] * len(arr)
X = np.vstack(X_parts).astype(np.float32)
labels = np.array(labels)
cats = np.unique(labels)
print(X.shape, len(cats), "categories")


/home/plb41586/app/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-25 16:35:18.061096: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-25 16:35:18.147253: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-25 16:35:19.688645: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightl

ValueError: need at least one array to concatenate

In [4]:

rng = np.random.default_rng(SEED)
N_MAX = 50_000
per_cat = max(1, N_MAX // len(cats))
idx = np.concatenate([rng.choice(np.where(labels == c)[0],
                                 min(per_cat, (labels == c).sum()), replace=False)
                      for c in cats])
X, labels = X[idx], labels[idx]

X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
X = StandardScaler().fit_transform(X)

# ---- reduce ----
N_NEIGHBORS = 300
MIN_DIST = 0.001
reducer = umap.UMAP(n_components=2, n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST,
                    metric="cosine")
Z = reducer.fit_transform(X)
print("UMAP done!")

# ---- interactive plot ----
fig = go.Figure()
for c in cats:
    m = labels == c
    fig.add_trace(go.Scattergl(
        x=Z[m, 0], y=Z[m, 1],
        mode="markers",
        name=str(c),
        marker=dict(size=3, opacity=0.6),
        hovertemplate=f"{c}<extra></extra>",
    ))

fig.update_layout(
    title=f"UMAP (n_neighbors={N_NEIGHBORS}, min_dist={MIN_DIST})",
    width=1000, height=800,
    legend=dict(itemsizing="constant", itemclick="toggle", itemdoubleclick="toggleothers"),
    xaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
    yaxis=dict(showticklabels=False, showgrid=False, zeroline=False,
               scaleanchor="x", scaleratio=1),
    template="plotly_white",
)

fig.write_html("embedding_projection.html", include_plotlyjs="cdn")
fig.show()

UMAP done!


In [ ]:
def load_data(data_dir, seed=0, n_max=50_000):
    rng = np.random.default_rng(seed)

    X_parts, labels = [], []
    for f in sorted(Path(data_dir).glob("*.npy")):
        arr = np.load(f)
        X_parts.append(arr)
        labels += [f.stem] * len(arr)
    X = np.vstack(X_parts).astype(np.float32)
    labels = np.array(labels)
    cats = np.unique(labels)
    print(X.shape, len(cats), "categories")

    per_cat = max(1, n_max // len(cats))
    idx = np.concatenate([
        rng.choice(np.where(labels == c)[0],
                   min(per_cat, (labels == c).sum()), replace=False)
        for c in cats
    ])
    X, labels = X[idx], labels[idx]

    X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    X = StandardScaler().fit_transform(X)

    return X, labels


def plot_umap(X, labels, n_neighbors=300, min_dist=0.001, out_path="embedding_projection.html"):
    reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=min_dist,
                        metric="correlation")
    Z = reducer.fit_transform(X)
    print("UMAP done!")

    cats = np.unique(labels)
    fig = go.Figure()
    for c in cats:
        m = labels == c
        fig.add_trace(go.Scattergl(
            x=Z[m, 0], y=Z[m, 1],
            mode="markers",
            name=str(c),
            marker=dict(size=3, opacity=0.6),
            hovertemplate=f"{c}<extra></extra>",
        ))
    fig.update_layout(
        title=f"UMAP (n_neighbors={n_neighbors}, min_dist={min_dist})",
        width=1000, height=800,
        legend=dict(itemsizing="constant", itemclick="toggle", itemdoubleclick="toggleothers"),
        xaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
        yaxis=dict(showticklabels=False, showgrid=False, zeroline=False,
                   scaleanchor="x", scaleratio=1),
        template="plotly_white",
    )
    fig.write_html(out_path, include_plotlyjs="cdn")
    fig.show()
    return fig

In [4]:
DATA_DIR = Path("Outputs/Embeddings/PacketEmbeddings_AutoEncoder/FirstTryUnknownState")
SEED = 42
X, labels = load_data(DATA_DIR, seed=SEED)
fig = plot_umap(X, labels)

(2202044, 32) 15 categories
UMAP done!


In [5]:

DATA_DIR = Path("Outputs/Embeddings/PacketEmbeddings_AutoEncoder/Untrained")
SEED = 42
X, labels = load_data(DATA_DIR, seed=SEED)
fig = plot_umap(X, labels)

(2202044, 64) 15 categories
UMAP done!


In [6]:

DATA_DIR = Path("Outputs/Embeddings/PacketEmbeddings_AutoEncoder/Epoch1")
SEED = 42
X, labels = load_data(DATA_DIR, seed=SEED)
fig = plot_umap(X, labels)

(2202044, 64) 15 categories
UMAP done!
